# Capítulo 02 — Simulación Estocástica de Monte Carlo 

**Proyecto:** Análisis Forense de Falla en Tubería HDPE (Ruta E-89)  
**Fase:** Reconstrucción Probabilística de Presiones ($P_{max}$ y $P_{min}$)  
**Fecha:** Diciembre 2025

---

## 1. Contexto y Objetivos Forenses

El Notebook 01 nos entregó un catálogo determinista de eventos. Sin embargo, el SCADA (muestreo a 1 min) sufre de **ceguera temporal**: es incapaz de registrar los picos transitorios que ocurren en milisegundos (golpes de ariete).

**Objetivo:** Utilizar el método de Monte Carlo para reconstruir la "historia oculta" de la presión. Transformamos cada evento registrado en 5,000 escenarios físicos posibles, incorporando la incertidumbre de las variables no observadas.

### 1.1 Metodología Bimodal (Dos Motores Físicos)
No todos los eventos se rigen por las mismas ecuaciones. Implementamos una lógica de ramificación física:

#### Ruta A: Fatiga por Maniobra (Joukowsky Probabilístico)
* **Eventos:** `FAST_START` (Arranque) y `FAST_STOP` (Parada).
* **Fundamento:** La sobrepresión máxima se rige por la Ecuación de Joukowsky, modulada por la atenuación de cierre lento (Michaud).
  $$ \Delta H = \frac{a \cdot \Delta V}{g} \cdot \phi(T_c) $$
* **Incertidumbre ($T_c$):** El tiempo de cierre es la variable crítica. Lo modelamos con una distribución **Log-Normal** para capturar la naturaleza asimétrica de la operación humana: muchas maniobras normales (moda en 30s) y raras maniobras bruscas (cola izquierda < 5s).

#### Ruta B: Colapso de Cavidad (Modelo Heurístico)
* **Eventos:** `LOW_PRESSURE` (Vacío/Censura).
* **Fundamento:** Cuando $P \le 0$, se forma una cavidad de vapor. El daño ocurre al re-encontrarse las columnas de agua (colapso), generando un pico secundario.
* **Modelo:** $P_{max} = P_0 + k \cdot \Delta P_{Joukowsky}$. 
  Donde $k$ es un factor de amplificación estocástico que crece con la duración del vacío (mayor volumen de vapor = colapso más violento).

### 1.2 Factor de Localización ($s_Q$)
Reconocemos una incertidumbre epistémica: el sensor de caudal puede estar lejos del punto de la maniobra. La señal se atenúa con la distancia. Introducimos un factor de escala $s_Q \ge 1$ para corregir esta subestimación:
$$ \Delta Q_{local} = s_Q \cdot \Delta Q_{obs} $$

---

## 2. Configuración del Entorno

In [14]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import os
import json
import logging
from scipy.stats import lognorm, uniform

# --- Configuración de Logging con Trazabilidad ---
logging.basicConfig(
    level=logging.INFO, 
    format='%(asctime)s - %(levelname)s - %(message)s', 
    force=True
)
logger = logging.getLogger("MonteCarlo_Engine")

sns.set_context("paper", font_scale=1.2)
plt.style.use('seaborn-v0_8-whitegrid')

# --- CONSTANTES FÍSICAS (Basadas en Ingeniería de Detalle) ---
# Estas constantes definen los límites físicos del sistema.
PHYSICS = {
    "gravity_g": 9.81,          # m/s^2
    "pipe_length_L": 2500.0,    # metros (Longitud tramo crítico)
    "diameter_D": 0.315,        # metros (DN315)
    "PN_nominal": 100.0,        # mca (Límite operativo PN10)
    "PN_ultimate": 125.0,       # mca (Límite de rotura con SF 1.25)
    # Parámetros para la Celeridad (a) del HDPE
    "wave_speed_mode": 350.0,   # m/s (Valor típico HDPE envejecido)
    "wave_speed_min": 250.0,    # m/s (Límite inferior)
    "wave_speed_max": 450.0     # m/s (Límite superior)
}

# Rutas de Archivos
INPUT_CATALOG = "../data/processed/catalogo_maniobras_clasificadas.parquet"
INPUT_MASTER  = "../data/processed/sensores_limpios.parquet"
OUTPUT_SIM    = "../data/processed/resultados_monte_carlo.parquet"

logger.info("Entorno inicializado. Parámetros físicos cargados.")

2025-12-29 01:43:11,212 - INFO - Entorno inicializado. Parámetros físicos cargados.


## 3. Carga de Datos y Auditoría de Cobertura

**Punto Crítico de Auditoría:** Es imperativo establecer qué eventos del catálogo estadístico (NB01) tienen la calidad suficiente para ser simulados físicamente. Filtramos el "ruido" para concentrarnos en la "señal".

In [15]:
# 1. Cargar Catálogo (Output del NB01)
if not os.path.exists(INPUT_CATALOG):
    raise FileNotFoundError("No se encuentra el catálogo de eventos clasificados.")
df_events = pd.read_parquet(INPUT_CATALOG)

# 2. Cargar Serie Maestra (Para estimación local de P0)
try:
    df_master = pd.read_parquet(INPUT_MASTER)
    if 'ts' in df_master.columns: df_master.set_index('ts', inplace=True)
    HAS_MASTER = True
    logger.info(f"Serie Maestra disponible: {len(df_master)} registros.")
except:
    HAS_MASTER = False
    logger.warning("Serie Maestra NO disponible. Se usará P0 Fallback.")

# 3. Mapeo Taxonómico (Traducción Estadística -> Física)
# Definimos explícitamente qué clases se simulan y cuáles se ignoran (Ruido).
TAXONOMY_MAP = {
    # --- RUTA 1: MANIOBRAS ---
    "FAST_START": "FAST_START", "START_UP": "FAST_START", 
    "FAST_STOP": "FAST_STOP", "SHUT_DOWN": "FAST_STOP", "PUMP_TRIP_BLACKOUT": "FAST_STOP",
    
    # --- RUTA 2: VACÍO ---
    "COLUMN_SEPARATION": "LOW_PRESSURE", "P_FLOOR_EPISODE": "LOW_PRESSURE",
    
    # --- EXCLUIDOS (RUIDO) ---
    # Excluimos OP_NOISE porque su energía es insuficiente para causar daño.
    # Excluimos DATA_GAP porque simular sin datos es especular.
    "OP_NOISE": "IGNORE", 
    "DATA_GAP_TRIP": "IGNORE", 
    "UNSURE": "IGNORE"
}

df_events['sim_class'] = df_events['event_class'].map(TAXONOMY_MAP).fillna("IGNORE")

# 4. Auditoría de Cobertura
print("\n--- AUDITORÍA DE INCLUSIÓN/EXCLUSIÓN DE EVENTOS ---")
print(f"Total Eventos Detectados: {len(df_events)}")
print(df_events['sim_class'].value_counts())

# Filtrado Final
simulation_target = df_events[df_events['sim_class'] != "IGNORE"].copy()
# Corrección de datos: Duración mínima 1 min para evitar errores de división
simulation_target['duration_min'] = simulation_target['duration_min'].replace(0, 1.0)

print(f"\n>>> Eventos Válidos para Simulación: {len(simulation_target)}")

2025-12-29 01:43:11,544 - INFO - Serie Maestra disponible: 475201 registros.



--- AUDITORÍA DE INCLUSIÓN/EXCLUSIÓN DE EVENTOS ---
Total Eventos Detectados: 730
sim_class
FAST_STOP       311
FAST_START      240
IGNORE          159
LOW_PRESSURE     20
Name: count, dtype: int64

>>> Eventos Válidos para Simulación: 571


## 4. Definición de Generadores Estocásticos

Aquí definimos las funciones de densidad de probabilidad (PDF) para nuestras variables desconocidas. 

**Rigor Técnico:** Para el tiempo de cierre ($T_c$), usamos una conversión de momentos para asegurar que, al variar la desviación estándar ($\sigma$), la **media** de la distribución se mantenga fija en el valor esperado (30s). Esto evita sesgar el promedio al aumentar la incertidumbre.

In [16]:
RNG = np.random.default_rng(42) # Semilla 42 para reproducibilidad total

def get_wave_speed(n):
    """
    Genera 'n' valores de Celeridad (a) [m/s].
    Distribución: Triangular (más probable en la moda, acotada por min/max).
    """
    return RNG.triangular(
        PHYSICS['wave_speed_min'], 
        PHYSICS['wave_speed_mode'], 
        PHYSICS['wave_speed_max'], 
        size=n
    )

def get_closing_time(n, mean_tc, sigma_ln):
    """
    Genera 'n' Tiempos de Cierre (Tc) [s].
    Distribución: Log-Normal parametrizada por Media Real.
    Justificación: Modela la asimetría de la operación humana (límite físico en 0).
    """
    # Conversión: Parametros mu/sigma de la normal subyacente para obtener la media deseada
    mu = np.log(mean_tc) - 0.5 * (sigma_ln ** 2)
    scale = np.exp(mu)
    return lognorm.rvs(s=sigma_ln, scale=scale, size=n, random_state=RNG)

def get_dq_scale(n, mean_scale=1.0, sigma_scale=0.35):
    """
    Genera 'n' Factores de Localización (sQ) [adimensional].
    Distribución: Log-Normal.
    Justificación: Corrige la atenuación de señal por distancia sensor-evento.
    """
    mu = np.log(mean_scale) - 0.5 * (sigma_scale ** 2)
    scale = np.exp(mu)
    return lognorm.rvs(s=sigma_scale, scale=scale, size=n, random_state=RNG)

def get_k_factor(n, duration_min, scenario):
    """
    Genera 'n' Factores de Amplificación de Colapso (k).
    Distribución: Uniforme condicionada por duración.
    Justificación: Heurística basada en literatura (Bergant et al.).
    """
    if scenario == 'Optimistic':
        return RNG.uniform(1.0, 1.5, size=n)
    elif scenario == 'Pessimistic':
        return RNG.uniform(1.5, 3.5, size=n)
    else: # Baseline
        # Si el vacío duró poco, el colapso es menor. Si duró mucho, k es mayor.
        if duration_min <= 5:
            return RNG.uniform(1.0, 2.0, size=n)
        else:
            return RNG.uniform(1.5, 3.0, size=n)

def get_p0_local(t_start, fallback=35.0):
    """
    Obtiene la Presión Estática Pre-Evento (P0) [mca].
    Metodología: Mediana de los 5 minutos previos al evento.
    """
    if not HAS_MASTER: return fallback, "Fallback_NoMaster"
    try:
        # Ventana de -5 a -1 minuto antes
        window = df_master.loc[t_start - pd.Timedelta(minutes=5) : t_start - pd.Timedelta(minutes=1)]
        if len(window) < 2: return fallback, "Fallback_Gap"
        p0_est = window['p_mca'].median()
        # Sanity check: P0 no puede ser negativa ni nula para un arranque
        if pd.isna(p0_est) or p0_est <= 0:
            return fallback, "Fallback_BadData"
        return p0_est, "Local_Data"
    except:
        return fallback, "Fallback_Error"

## 5. Simulación Multiescenario (El Motor Físico)

Ejecutamos el análisis de sensibilidad variando la incertidumbre operativa ($\{sigma$).

**Parámetros de Ejecución:**
* **N = 5000 iteraciones:** Necesario para asegurar la estabilidad estadística de los percentiles extremos (P99, P01).
* **Escenarios:** `Baseline` (Realista) y `Pessimistic` (Estresado).

In [17]:
SCENARIOS = {
    "Baseline":    {"sigma_tc": 0.7},
    "Pessimistic": {"sigma_tc": 1.0}
}

N_ITERS = 5000
AREA = np.pi * (PHYSICS["diameter_D"] / 2)**2

all_results = []

logger.info(f"Iniciando Loop de Simulación (N={N_ITERS} por evento)...")

for sc_name, params in SCENARIOS.items():
    logger.info(f"---> Procesando Escenario: {sc_name}")
    
    for idx, row in simulation_target.iterrows():
        # --- PASO 1: Datos Determinísticos (La Evidencia) ---
        t_start = row['t_start']
        sim_class = row['sim_class']
        duration = row['duration_min']
        # Usamos abs() porque la dirección (signo) la maneja la clase del evento
        dQ_ls = abs(row.get('max_dQ_ls', 10.0))
        
        # Obtenemos la condición inicial real del sistema
        P0, p0_src = get_p0_local(t_start, fallback=35.0)
        
        # --- PASO 2: Muestreo Estocástico (La Incertidumbre) ---
        # Generamos 5000 universos paralelos para este evento
        a_vec = get_wave_speed(N_ITERS)
        tc_vec = get_closing_time(N_ITERS, mean_tc=30.0, sigma_ln=params['sigma_tc'])
        sQ_vec = get_dq_scale(N_ITERS)
        
        # --- PASO 3: Física de Fluidos (Las Ecuaciones) ---
        # 3.1 Tiempo Crítico: Umbral entre cierre rápido y lento
        tcrit_vec = (2 * PHYSICS['pipe_length_L']) / a_vec
        
        # 3.2 Atenuación de Michaud: Si Tc > Tcrit, el golpe se reduce
        michaud_factor = np.clip(tcrit_vec / tc_vec, 0, 1.0)
        
        # 3.3 Joukowsky Potencial: Energía máxima disponible
        dV = (dQ_ls / 1000.0) / AREA
        dH_joukowsky = (a_vec * dV * sQ_vec) / PHYSICS['gravity_g']
        
        # --- PASO 4: Lógica de Ramificación por Tipo de Evento ---
        
        if sim_class == 'FAST_STOP':
            # Parada: El golpe se suma a la presión base
            P_max_vec = P0 + (dH_joukowsky * michaud_factor)
            P_min_vec = np.full(N_ITERS, P0) # Asumimos no depresión significativa
            
        elif sim_class == 'FAST_START':
            # Arranque: El golpe se resta a la presión base (Depresión)
            P_min_vec = P0 - (dH_joukowsky * michaud_factor)
            P_max_vec = np.full(N_ITERS, P0) # Rebote elástico conservador
            
        elif sim_class == 'LOW_PRESSURE':
            # Vacío: El daño viene del retorno (k * Joukowsky)
            k_vec = get_k_factor(N_ITERS, duration, sc_name)
            P_max_vec = P0 + (dH_joukowsky * k_vec)
            # Confirmamos la presión negativa durante el evento
            P_min_vec = np.minimum(0.0, P0 - dH_joukowsky)
            
        # --- PASO 5: Reducción Estadística (KPIs) ---
        # Calculamos la probabilidad de falla para este evento específico
        prob_fail_PN10 = np.mean(P_max_vec > PHYSICS['PN_nominal'])
        prob_fail_Ult = np.mean(P_max_vec > PHYSICS['PN_ultimate'])
        prob_vacuum = np.mean(P_min_vec < 0.0)
        
        all_results.append({
            "event_id": idx,
            "t_start": t_start,
            "scenario": sc_name,
            "sim_class": sim_class,
            "duration_min": duration,
            "P0_est": P0,
            "P0_source": p0_src,
            
            # Percentiles Clave (P50, P95, P99)
            "Pmax_p50": np.percentile(P_max_vec, 50),
            "Pmax_p95": np.percentile(P_max_vec, 95),
            "Pmax_p99": np.percentile(P_max_vec, 99),
            "Pmin_p01": np.percentile(P_min_vec, 1),
            
            # Probabilidades de Excedencia
            "Prob_Exceed_PN10": prob_fail_PN10,
            "Prob_Exceed_1.25PN": prob_fail_Ult,
            "Prob_Below_Zero": prob_vacuum
        })

df_res = pd.DataFrame(all_results)
logger.info(f"Simulación Finalizada. {len(df_res)} registros generados.")

2025-12-29 01:43:13,061 - INFO - Iniciando Loop de Simulación (N=5000 por evento)...
2025-12-29 01:43:13,066 - INFO - ---> Procesando Escenario: Baseline
2025-12-29 01:43:14,091 - INFO - ---> Procesando Escenario: Pessimistic
2025-12-29 01:43:15,169 - INFO - Simulación Finalizada. 1142 registros generados.


## 6. Resultados y Validación Forense

A continuación, presentamos la evidencia sintética generada. Identificamos los eventos que, bajo incertidumbre, mostraron alta probabilidad de violar los límites de diseño de la tubería.

In [18]:
# 6.1 Resumen Ejecutivo (Tabla de Riesgos)
summary = df_res.groupby('scenario').agg(
    Total_Events=('event_id', 'count'),
    Events_Risk_PN10=('Prob_Exceed_PN10', lambda x: (x > 0.01).sum()),
    Events_Risk_Vacuum=('Prob_Below_Zero', lambda x: (x > 0.01).sum()),
    Max_Peak_Pressure=('Pmax_p99', 'max')
).reindex(['Baseline', 'Pessimistic'])

print("\n--- RESUMEN DE RIESGO ESTRUCTURAL ---")
display(summary)

# 6.2 Top 10 Killer Events (Sobrepresión)
print("\n--- TOP 10 EVENTOS CRÍTICOS (SOBREPRESIÓN > PN10) ---")
baseline = df_res[df_res['scenario']=='Baseline']
top_max = baseline.sort_values('Pmax_p99', ascending=False).head(10)
display(top_max[['t_start', 'sim_class', 'Pmax_p99', 'Prob_Exceed_PN10']])

# 6.3 Top 10 Vacuum Events (Succión)
print("\n--- TOP 10 EVENTOS CRÍTICOS (SUCCIÓN / VACÍO) ---")
top_min = baseline.sort_values('Pmin_p01', ascending=True).head(10)
display(top_min[['t_start', 'sim_class', 'Pmin_p01', 'Prob_Below_Zero']])


--- RESUMEN DE RIESGO ESTRUCTURAL ---


,Total_Events,Events_Risk_PN10,Events_Risk_Vacuum,Max_Peak_Pressure
scenario,,,,
Baseline,571,3,212,170.681419
Pessimistic,571,5,228,192.232260



--- TOP 10 EVENTOS CRÍTICOS (SOBREPRESIÓN > PN10) ---


,t_start,sim_class,Pmax_p99,Prob_Exceed_PN10
183,2025-04-16 06:49:00,LOW_PRESSURE,170.681419,0.2062
494,2025-10-12 02:02:00,LOW_PRESSURE,147.373883,0.0938
101,2025-02-26 06:10:00,LOW_PRESSURE,105.158172,0.0160
134,2025-03-25 05:00:00,LOW_PRESSURE,93.391846,0.0046
133,2025-03-25 00:52:00,FAST_STOP,79.514855,0.0016
182,2025-04-16 02:00:00,FAST_STOP,77.516179,0.0010
548,2025-11-08 12:02:00,FAST_STOP,75.698043,0.0008
359,2025-07-16 01:58:00,LOW_PRESSURE,75.071785,0.0012
356,2025-07-15 22:31:00,FAST_STOP,73.728543,0.0004
221,2025-05-05 21:45:00,FAST_STOP,73.153235,0.0004



--- TOP 10 EVENTOS CRÍTICOS (SUCCIÓN / VACÍO) ---


,t_start,sim_class,Pmin_p01,Prob_Below_Zero
222,2025-05-06 01:00:00,FAST_START,-48.045167,0.5036
101,2025-02-26 06:10:00,LOW_PRESSURE,-46.934961,0.9778
494,2025-10-12 02:02:00,LOW_PRESSURE,-46.727452,0.9920
183,2025-04-16 06:49:00,LOW_PRESSURE,-46.672334,0.9160
477,2025-10-02 21:46:00,FAST_START,-42.836985,0.5710
134,2025-03-25 05:00:00,LOW_PRESSURE,-41.278312,0.9736
541,2025-10-27 21:22:00,FAST_START,-38.195380,0.5458
181,2025-04-16 00:30:00,FAST_START,-36.234055,0.4452
188,2025-04-23 00:48:00,FAST_START,-35.627824,0.4098
359,2025-07-16 01:58:00,LOW_PRESSURE,-30.828996,0.9344


## 7. Conclusiones del Capítulo 02

1.  **Dominio del Vacío:** El análisis confirma que los eventos de `LOW_PRESSURE` (Colapso de Cavidad) son los únicos capaces de generar presiones extremas (~145 mca) con alta probabilidad, dominando el riesgo de rotura catastrófica.
2.  **Riesgo de Succión:** Se valida la hipótesis de fatiga por sobalamiento. Existe una alta probabilidad de que la tubería opere en presión negativa durante los arranques bruscos y los episodios de vacío.
3.  **Seguridad Relativa de Maniobras:** Las paradas rápidas (`FAST_STOP`), aunque frecuentes, raramente superan la presión nominal (PN10) bajo condiciones estándar. Su rol es más probable como agentes de fatiga (propagación de grieta) que de iniciación de falla.

**Siguiente Paso (Notebook 03):** Integrar estos resultados en un modelo de **Daño Acumulado (Miner)** para estimar cuánta vida útil ha perdido la tubería.

In [19]:
df_res.to_parquet(OUTPUT_SIM)
logger.info(f"Resultados guardados exitosamente en: {OUTPUT_SIM}")

2025-12-29 01:43:15,238 - INFO - Resultados guardados exitosamente en: ../data/processed/resultados_monte_carlo.parquet
